# Day 012：MiniMind 原生推理调用链

本 Notebook 与 Day 012 互动档案配套。目标是把已经按源码顺序读过的原生 MiniMind 推理路径整理成可复现的小实验：

运行命令 -> init_model -> MiniMindForCausalLM -> MiniMindModel -> MiniMindBlock -> Attention -> logits -> 采样 -> KV Cache -> 返回完整 token 序列 -> decode 新回答。

Notebook 中的 shape 实验不代替源码阅读。每个 Cell 先看说明，再运行，并用自己的话解释输出。

## 1. 本日真实运行场景

使用的命令：

    python eval_llm.py --load_from model --device cpu --max_new_tokens 64 --temperature 0.1 --top_p 0.9 --show_speed 1

本次实际配置：

    batch_size = 1
    prompt token 数 = 21
    hidden_size = 768
    num_hidden_layers = 8
    num_attention_heads = 8
    num_key_value_heads = 4
    head_dim = 96
    vocab_size = 6400
    use_cache = True
    do_sample = True

原生 checkpoint 是 out/full_sft_768.pth。下面的实验尽量只依赖 PyTorch，
不在 Notebook 中自动加载完整模型，避免把“运行成功”误当成“已经理解”。

In [ ]:
import torch

prompt_len = 21
generated_len = 30
full_len = prompt_len + generated_len

print("prompt 长度：", prompt_len)
print("新生成长度：", generated_len)
print("完整序列长度：", full_len)

assert full_len == 51
print("generated_ids.shape 应为：", (1, full_len))
print("generated_ids[0][prompt_len:].shape 应为：", (generated_len,))

## 2. Attention 输出：Head 维度重新拼回 hidden_size

单 token 阶段，V 汇总后是：

    [batch, heads, query_len, head_dim] = [1, 8, 1, 96]

transpose(1, 2) 后：

    [1, 1, 8, 96]

reshape(bsz, seq_len, -1) 后：

    [1, 1, 768]

第一次处理 21 个 prompt token 时对应：

    [1, 8, 21, 96] -> [1, 21, 8, 96] -> [1, 21, 768]

In [ ]:
batch, heads, seq_len, head_dim = 1, 8, 3, 4
output = torch.arange(batch * heads * seq_len * head_dim).reshape(
    batch, heads, seq_len, head_dim
)

print("Attention 内部 shape：", output.shape)
transposed = output.transpose(1, 2)
print("transpose(1, 2) 后：", transposed.shape)
combined = transposed.reshape(batch, seq_len, heads * head_dim)
print("reshape 后：", combined.shape)

assert tuple(transposed.shape) == (1, 3, 8, 4)
assert tuple(combined.shape) == (1, 3, 32)

## 3. 两条残差和 FeedForward shape

一个 MiniMindBlock 的主线：

    x
    -> input_layernorm
    -> Attention
    -> Attention 输出 + x
    -> post_attention_layernorm
    -> FeedForward（768 -> 2432 -> 768）
    -> FeedForward 输出 + 上一步结果

普通 FeedForward 中间分支的 shape：

    x                         [1, 21, 768]
    gate_proj(x)              [1, 21, 2432]
    up_proj(x)                [1, 21, 2432]
    逐元素相乘                [1, 21, 2432]
    down_proj(...)             [1, 21, 768]

两个残差相加都保持 [1, 21, 768]，不会改变 token 数量。

In [ ]:
batch, seq_len, hidden_size = 1, 21, 768
intermediate_size = 2432

gate = torch.empty(batch, seq_len, intermediate_size)
up = torch.empty(batch, seq_len, intermediate_size)
mlp_out = torch.empty(batch, seq_len, hidden_size)

print("gate_proj / up_proj：", gate.shape, up.shape)
print("down_proj 输出：", mlp_out.shape)
assert tuple(gate.shape) == (1, 21, 2432)
assert tuple(mlp_out.shape) == (1, 21, 768)

## 4. KV Cache 的时间线

缓存属于每一层，所以有 8 个缓存槽位。每个槽位是一对 K/V：

    第 0 层 K/V -> presents[0]
    ...
    第 7 层 K/V -> presents[7]

第一次 prompt 计算完成后，Cache 长度是 21。虽然随后 input_ids
会先拼成 22，但新 token 还没有经过下一轮 Attention，因此 Cache
仍然是 21。第二次 forward 只输入最新的一个 token，返回后 Cache
才增长为 22。

In [ ]:
timeline = [
    ("第一次 forward 前", 21, 21, 0),
    ("第一次 forward 后", 21, 0, 21),
    ("追加 next_token 后", 22, 0, 21),
    ("第二次 forward 输入切片", 22, 1, 21),
    ("第二次 forward 后", 22, 0, 22),
]

print("时刻 | 完整 input_ids 长度 | 本轮送入 token 数 | Cache 长度")
for row in timeline:
    print(" | ".join(map(str, row)))

assert timeline[2][-1] == 21
assert timeline[4][-1] == 22

## 5. logits 到 next_token

MiniMindForCausalLM.forward() 的输出：

    hidden_states [1, 21, 768]
    lm_head       768 -> 6400
    logits        [1, 21, 6400]

generate() 只取最后一个位置：

    logits[:, -1, :] -> [1, 6400]

这 6400 个数是词表中 6400 个候选 token 的原始分数，不是概率。
当前 do_sample=True，所以后续是：

    温度 -> repetition penalty -> Top-K -> Top-P
    -> Softmax -> multinomial -> next_token [1, 1]

In [ ]:
logits = torch.tensor([[2.0, 1.0, 0.0, -3.0]])
temperature = 0.5
scaled = logits / temperature
probabilities = torch.softmax(scaled, dim=-1)

print("原始 logits：", logits)
print("温度调整后：", scaled)
print("概率：", probabilities)
print("概率和：", probabilities.sum(dim=-1))

assert tuple(scaled.shape) == (1, 4)
assert torch.allclose(probabilities.sum(dim=-1), torch.ones(1))

## 6. EOS、返回和外层 decode

生成循环结束有两种原因：

    生成 EOS -> finished.all() -> break
    没有 EOS -> 执行满 max_new_tokens

默认 return input_ids 返回完整序列：

    prompt token + 新生成 token

eval_llm.py 再用：

    generated_ids[0][len(inputs["input_ids"][0]):]

切掉 prompt，只把新生成 token 交给 tokenizer.decode。对于 prompt 长度
21、生成 30 个 token 的例子，切片后的 shape 是 [30]，完整张量是 [1, 51]。

In [ ]:
prompt_ids = torch.arange(21)
new_ids = torch.arange(100, 130)
generated_ids = torch.cat([prompt_ids, new_ids]).unsqueeze(0)

prompt_length = len(prompt_ids)
new_part = generated_ids[0][prompt_length:]

print("generated_ids.shape：", generated_ids.shape)
print("原始 prompt shape：", prompt_ids.shape)
print("新生成部分 shape：", new_part.shape)
print("新生成部分首尾：", new_part[:3], new_part[-3:])

assert tuple(generated_ids.shape) == (1, 51)
assert tuple(new_part.shape) == (30,)

## 7. 回到源码时的定位表

源码文件：

    model/model_minimind.py

本日已经按调用关系读到：

    Attention.forward()
    -> MiniMindBlock.forward()
    -> MiniMindModel.forward()
    -> MiniMindForCausalLM.forward()
    -> MiniMindForCausalLM.generate()

下一学习日先复核最后一个 decode 切片 shape，再用轻量追踪实验打印
第一次 prompt 和第二次单 token 调用的真实 shape，最后回到 eval_llm.py
的 conversation.append 和速度统计。之后再决定是否进入
trainer/train_full_sft.py 的训练入口。

In [ ]:
from pathlib import Path

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path("/home/zcf/githubs/minimind/minimind"),
]
project_root = next(
    path for path in candidates
    if (path / "model" / "model_minimind.py").exists()
)
source_path = project_root / "model" / "model_minimind.py"

print("项目目录：", project_root)
print("源码文件：", source_path)

source_lines = source_path.read_text(encoding="utf-8").splitlines()
for start, end in [(90, 135), (178, 232), (234, 275)]:
    print(f"\n--- lines {start}:{end} ---")
    for number in range(start, min(end, len(source_lines)) + 1):
        print(f"{number:>4}: {source_lines[number - 1]}")

## 自测题（先独立回答，再回到对话中验收）

1. prompt 长度 21、生成 30 个 token 时，generated_ids.shape 是什么？
2. generated_ids[0][21:] 的 shape 是什么？
3. 第一次 forward 完成后，Cache 为什么是 21 而不是 22？
4. logits 的最后一维 6400 是什么？
5. 单 token 解码时，Q、K、V 对齐 Head 后的 shape 分别是什么？

本 Notebook 只提供可运行材料；答案和源码中的下一步在 Day 13
互动学习时逐题确认。